# Meta-analysis
Following parallelized plink2 GLM across ancestries and chromosomes, followed by results concatenation and gwaslab plots.

In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import gwaslab as gl

## Configuration

In [ ]:
wd = "/path/to/home"

ANCESTRIES   = ["CAS", "MDE", "EUR"]
CHROMOSOMES  = list(range(1, 23))                      # 1–22; edit as needed
PHENOTYPE    = "DISEASE"                               # pheno name in covariate file
LINEAR_METRIC = None                                   # unused in logistic; kept for signature parity

MAX_WORKERS = 2   # parallelism across (ancestry, chromosome) pairs
plink_threads = 1 # threads given to each plink2 call

```bash
for a in MDE EUR CAS ; do plink2 --pfile CATPD_before_qc_${a} --keep CATPD_unrel.keep --maf 0.01 --indep-pairwise 200 50 0.2 --out pruned_${a} ; plink2 --pfile CATPD_before_qc_${a} --keep CATPD_unrel.keep --extract pruned_${a}.prune.in --make-pgen --out CATPD_pruned_${a} ; plink2 --pfile CATPD_pruned_${a} --pca 10 --out CATPD_pruned_${a}_PC10 ; done &
```

In [ ]:
key = pd.read_csv('/path/to/home/CATPD_unrelated.cov', sep='\t', header=0)

for a in ANCESTRIES:
    pca = pd.read_csv(f'CATPD_pruned_{a}_PC10.eigenvec', sep='\t', header=0)
    pca = pca.loc[:, ["#IID"] + [f"PC{i}" for i in range(1, 11)]]
    merged = pd.merge(key, pca, on="#IID", how='right')
    merged['#IID'] = "0_" + merged['#IID'].astype(str)
    merged['DISEASE'] = merged['DISEASE'] + 1
    merged = merged[['#IID', 'DISEASE', 'SEX', 'AGE_AAO', 'PC1', 'PC2', 'PC3', 'PC4', 'PC5', 'PC6', 'PC7', 'PC8', 'PC9', 'PC10']]
    merged.to_csv(f'{a}_unrelated.cov', sep='\t', index=False)
    print(merged.head())

### MR-MEGA

In [ ]:
def format_for_mrmega(ancestry):
    # Plink glm output per ancestry
    infile = f"{WD}/{ancestry}/{ancestry}_all_chr.DISEASE.glm.logistic.hybrid"
    df = pd.read_csv(infile, sep="\t", low_memory=False)

    out = pd.DataFrame({
        "MARKERNAME": df["ID"],
        "EA":         df["ALT"],
        "NEA":        df["REF"],
        "OR":         np.exp(df["BETA"]),
        "OR_95L":     np.exp(df["L95"]),
        "OR_95U":     np.exp(df["U95"]),
        "EAF":        df["A1_FREQ"],
        "N":          df["OBS_CT"],
        "CHROMOSOME": df["#CHROM"],
        "POSITION":   df["POS"],
    }).dropna()

    outpath = f"{WD}/MRMEGA_Results/{ancestry}_mrmega_input.txt"
    out.to_csv(outpath, sep="\t", index=False)
    print(f"[{ancestry}] {len(out):,} variants written in {outpath}")
    return outpath

In [ ]:
# Format all three
import os
os.makedirs(f"{WD}/MRMEGA_Results", exist_ok=True)

In [ ]:
import numpy as np
input_list = f"{WD}/MRMEGA_Results/input_list.txt"
with open(input_list, "w") as f:
    for anc in ANCESTRIES:
        path = format_for_mrmega(anc)
        f.write(path + "\n")

In [ ]:
# Run MR-MEGA
subprocess.run([
    f"{WD}/MRMEGA/MR-MEGA",
    "-i",    input_list,
    "--pc",  "1",         
    "--out", f"{WD}/MRMEGA_Results/CATPD_MRMEGA",
], check=True)

## Manhattan + QQ per ancestry

In [ ]:
for ancestry in ANCESTRIES:
    print(f"\n── {ancestry} ──")
    df_file = f'/data/hestia/misayan/CATPD_GWAS/subset_gwas/{ancestry}/{ancestry}_all_chr.DISEASE.glm.logistic.hybrid'
    df = pd.read_csv(df_file, header=0, sep='\t')

    mysumstats = gl.Sumstats(
        df,
        snpid  = "ID",
        chrom  = "#CHROM",
        pos    = "POS",
        ea     = "A1",
        nea    = "OMITTED",
        beta    = "BETA",
        se     = "SE",
        p      = "P",
        n      = "OBS_CT",
        build  = "38",    
    )

    # Manhattan plot
    mysumstats.plot_mqq(
        mode       = "mqq",
        sig_level  = 5e-8,
    )

### METAL

In [ ]:
ANCESTRIES = ["CAS", "MDE", "EUR"]

# Make METAL script
metal_script = f"{WD}/METAL/metal_script.txt"
metal_out    = f"{WD}/METAL/CATPD_meta"
os.makedirs(f"{WD}/METAL", exist_ok=True)

with open(metal_script, "w") as f:
    f.write("SCHEME STDERR\n")       
    f.write("AVERAGEFREQ ON\n")
    f.write("MINMAXFREQ ON\n")

    for anc in ANCESTRIES:
        infile = f"{WD}/{anc}/{anc}_all_chr.DISEASE.glm.logistic.hybrid"
        f.write(f"MARKER   ID\n")
        f.write(f"ALLELE   ALT REF\n")
        f.write(f"FREQ     A1_FREQ\n")
        f.write(f"EFFECT   BETA\n")
        f.write(f"STDERR   SE\n")
        f.write(f"PVALUE   P\n")
        f.write(f"WEIGHT   OBS_CT\n")
        f.write(f"PROCESS  {infile}\n\n")

    f.write(f"OUTFILE  {metal_out} .tbl\n")
    f.write("ANALYZE HETEROGENEITY\n")  
    f.write("QUIT\n")

subprocess.run(["metal", metal_script], check=True)
print(f"METAL: {metal_out}1.tbl")

In [ ]:
df = pd.read_csv(f'{metal_out}1.tbl', header=0, sep='\t')
df = df[~df['Direction'].str.contains('?', regex=False, na=False)]
df[['#CHROM', 'POS', 'REF' ,'ALT']] = df['MarkerName'].str.split(':', expand=True)

# MarkerName      Allele1 Allele2 Freq1   FreqSE  MinFreq MaxFreq Effect  StdErr  log(P)  Direction       HetISq  HetChiSq        HetDf   logHetP
mysumstats = gl.Sumstats(
    df,
    snpid  = "MarkerName",
    chrom  = "#CHROM",
    pos    = "POS",
    ea     = "Allele1",
    nea    = "Allele2",
    beta   = "Effect",
    se     = "StdErr",
    p      = "P-value",
    direction="Direction",
    build  = "38", 
)

# Manhattan plot
mysumstats.plot_mqq(
    mode       = "mqq",
    sig_level  = 5e-8,
    anno       = 'GENENAME')

### GWAMA

In [ ]:
def format_for_gwama(ancestry):
    infile = f"{WD}/{ancestry}/{ancestry}_all_chr.DISEASE.glm.logistic.hybrid"
    df = pd.read_csv(infile, sep="\t", low_memory=False)

    out = pd.DataFrame({
        "MARKERNAME": df["ID"],
        "EA":         df["ALT"],
        "NEA":        df["REF"],
        "OR":         np.exp(df["BETA"]),
        "OR_95L":     np.exp(df["L95"]),
        "OR_95U":     np.exp(df["U95"]),
        "SE":         df["SE"],
        "EAF":        df["A1_FREQ"],
        "N":          df["OBS_CT"],
        "N_cases":    (df["A1_CASE_CT"] / 2).round().astype(int),
        "N_controls": (df["A1_CTRL_CT"] / 2).round().astype(int),
    }).dropna()

    outpath = f"{WD}/GWAMA/{ancestry}_gwama_input.txt"
    out.to_csv(outpath, sep="\t", index=False)
    print(f"[{ancestry}] {len(out):,} variants written in {outpath}")
    return outpath

In [ ]:
# Format all ancestries
os.makedirs(f"{WD}/GWAMA", exist_ok=True)
input_list = f"{WD}/GWAMA/input_list.txt"
with open(input_list, "w") as f:
    for anc in ANCESTRIES:
        path = format_for_gwama(anc)
        f.write(path + "\n")

In [ ]:
# GWAMA FIXED
subprocess.run([
    "GWAMA",
    "-i",  input_list,
    "-o",  f"{WD}/GWAMA/CATPD_GWAMA_Fixed",
    # "-r",               # random effects
    # "--quantitative",   # remove this line for binary trait
    "-gc",              # genomic control correction
    # "--no_allele_check" # if alleles differ across ancestries
], check=True)

# GWAMA RANDOM
subprocess.run([
    "GWAMA",
    "-i",  input_list,
    "-o",  f"{WD}/GWAMA/CATPD_GWAMA_Random",
    "-r",               # random effects
    # "--quantitative",   # remove this line for binary trait
    "-gc",              # genomic control correction
    # "--no_allele_check" # if alleles differ across ancestries
], check=True)

In [ ]:
df_file = 'CATPD_GWAMA.out'
df = pd.read_csv(df_file, header=0, sep='\t')
df[['#CHROM', 'POS', 'REF' ,'ALT']] = df['rs_number'].str.split(':', expand=True)
# MarkerName      Allele1 Allele2 Freq1   FreqSE  MinFreq MaxFreq Effect  StdErr  log(P)  Direction       HetISq  HetChiSq        HetDf   logHetP
mysumstats = gl.Sumstats(
    df,
    snpid  = "rs_number",
    chrom  = "#CHROM",
    pos    = "POS",
    ea     = "other_allele",
    nea    = "reference_allele",
    OR   = "OR",
    se     = "OR_se",
    p      = "p-value",
    direction="effects",
    build  = "38", 
)

# Manhattan plot
mysumstats.plot_mqq(
    mode       = "mqq",
    sig_level  = 5e-8,
    anno       = True)